# Binary domain vs rest (v5)

Same `real_multiclass_v5.csv`; label **domain=1**, all news topics **rest=0**.

Grouped split by `doc_id`. Includes dummy baselines + TF-IDF/LR.

Multiclass counterpart: [`ml_multi_v5.ipynb`](ml_multi_v5.ipynb).

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score,
    average_precision_score,
    RocCurveDisplay,
    f1_score,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
def resolve_data_path(filename: str = "real_multiclass_v5.csv") -> Path:
    candidates = [
        Path.cwd() / "data" / filename,
        Path.cwd() / "../data" / filename,
        Path.cwd().parent / "data" / filename,
    ]
    for path in candidates:
        if path.resolve().exists():
            return path.resolve()
    raise FileNotFoundError(
        f"Could not find {filename}. Tried:\n" + "\n".join(str(p.resolve()) for p in candidates)
    )

DATA_PATH = resolve_data_path()
print("Loading:", DATA_PATH)


## 2. Load + binary label

In [ ]:
df = pd.read_csv(DATA_PATH)
df["y"] = (df["topic"] == "domain").astype(int)

print("shape:", df.shape)
print("binary balance:")
print(df["y"].value_counts().rename({0: "rest", 1: "domain"}))


## 3. Grouped split

In [ ]:
def grouped_train_test_split(df, test_size=0.2, random_state=42):
    """Hold out whole doc_id groups; stratify on binary label at doc level."""
    doc_meta = df.groupby("doc_id", as_index=False).agg(y=("y", "first"))
    train_docs, test_docs = train_test_split(
        doc_meta["doc_id"],
        test_size=test_size,
        stratify=doc_meta["y"],
        random_state=random_state,
    )
    test_set = set(test_docs)
    train_df = df.loc[~df["doc_id"].isin(test_set)].copy()
    test_df = df.loc[df["doc_id"].isin(test_set)].copy()
    return train_df, test_df

train_df, test_df = grouped_train_test_split(df, test_size=0.2, random_state=RANDOM_STATE)
X_train, y_train = train_df["text"], train_df["y"]
X_test, y_test = test_df["text"], test_df["y"]

print(f"train rows: {len(train_df):,}  test rows: {len(test_df):,}")
print(f"train docs: {train_df['doc_id'].nunique():,}  test docs: {test_df['doc_id'].nunique():,}")
print("\ntrain label counts:")
print(y_train.value_counts().sort_index())
print("\ntest label counts:")
print(y_test.value_counts().sort_index())


## 4. Baselines + tuned LR

In [ ]:
baselines = {
    "majority": DummyClassifier(strategy="most_frequent"),
    "stratified": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
}

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True)),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)),
])

param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "tfidf__max_features": [5000, 10000],
    "clf__C": [0.1, 1.0, 10.0],
}

search = GridSearchCV(pipe, param_grid, scoring="f1", cv=5, n_jobs=-1, refit=True, verbose=1)
search.fit(X_train, y_train)
best_model = search.best_estimator_
print("best CV F1:", round(search.best_score_, 4))
print("best params:", search.best_params_)


## 5. Holdout metrics

In [ ]:
for name, est in baselines.items():
    est.fit(X_train, y_train)
    pred = est.predict(X_test)
    print(f"=== {name} ===")
    print(classification_report(y_test, pred, target_names=["rest", "domain"], digits=3))

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
y_score = best_model.predict_proba(X_test)[:, 1]

print("=== TF-IDF + LR (tuned) ===")
print(classification_report(y_test, y_pred, target_names=["rest", "domain"], digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_score), 4))
print("average precision:", round(average_precision_score(y_test, y_score), 4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axes[0], cmap="Blues")
axes[0].set_title("Confusion matrix")
RocCurveDisplay.from_predictions(y_test, y_score, ax=axes[1])
axes[1].set_title("ROC")
plt.tight_layout()
plt.show()


## 6. Save model

In [ ]:
import joblib

def resolve_models_dir() -> Path:
    for base in [Path.cwd(), Path.cwd().parent]:
        d = base / "data" / "models"
        if d.parent.joinpath("real_multiclass_v5.csv").exists() or (base / "data").exists():
            d.mkdir(parents=True, exist_ok=True)
            return d.resolve()
    raise FileNotFoundError("Could not resolve data/models directory")

MODELS_DIR = resolve_models_dir()
MODEL_PATH = MODELS_DIR / "ml_binary_domain_v5.joblib"
joblib.dump(best_model, MODEL_PATH)
print("saved:", MODEL_PATH)
